# 03 — Exploration PCA, stabilité et shortlist (tâches 15–16)

Ce notebook utilise exclusivement les objets QC acceptés des batches 1–2 et les matrices/prétraitements techniquement acceptés par le notebook 02. Les batches 3–4 ne sont ni ajustés, ni projetés, ni évalués ici.

La sélection porte uniquement sur les prétraitements, séparément pour `object_matrix` et `pixel_matrix`. Après la revue humaine, chaque prétraitement doit rester admissible sur toutes les variantes de matrice attendues ; ses métriques sont agrégées en pire cas, puis tous les prétraitements du front de Pareto sont conservés, sans score pondéré, filtre de diversité ni plafond actif.


In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR if (CURRENT_DIR / "src").is_dir() else CURRENT_DIR.parent
if not (PROJECT_ROOT / "src").is_dir():
    raise RuntimeError("Launch the notebook from the project root or notebooks/.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import experiment_config as expcfg
from src.io.database_h5 import load_nir_uco_h5
from src.protocol_governance import sha256_file, verify_frozen_protocol
from src.spectra.band_selection import select_wavelength_range_from_database
from src.utils import save_parquet
from src.workflows.matrix_preprocessing import wavelength_axis_id
from src.workflows.pca import (
    build_pca_candidate_plan,
    evaluate_pca_stability,
    fit_pca_candidate,
    subset_object_db_for_pca,
    summarize_pca_stability,
)
from src.workflows.pca_selection import (
    PCA_TECHNICAL_FLAG_COLUMNS,
    apply_pca_artifact_review_decisions,
    build_pca_artifact_review_table,
    build_pca_run_fingerprint,
    build_pca_scoring_diagnostics,
    build_pca_selection_diagnostics,
    freeze_pca_shortlist,
    hash_pca_input_artifacts,
    hash_pca_review_table,
    pca_input_artifact_paths,
    select_pca_preprocessing_shortlist,
    validate_pca_artifact_review,
    validate_pca_preprocessing_shortlist,
)
from src.workflows.protocol_audit import assert_no_forbidden_score_columns
from src.workflows.protocol_split import build_grouped_folds, eligible_object_ids
from src.visualization.plot_pca import build_pca_visual_review_pdf

pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", expcfg.PCA_MAX_ROWS_TO_DISPLAY)


## 1. Protocole gelé et artefacts d’entrée

In [2]:
RESULTS_TAG = (
    f"{int(expcfg.WAVELENGTH_WINDOW_MIN_NM)}_{int(expcfg.WAVELENGTH_WINDOW_MAX_NM)}"
    if expcfg.USE_WAVELENGTH_WINDOW
    else expcfg.DEFAULT_RESULTS_TAG
)
PROTOCOL_DIR = PROJECT_ROOT.joinpath(*expcfg.PROTOCOL_ARTIFACT_RELATIVE_DIR)
protocol_checks_df = verify_frozen_protocol(PROTOCOL_DIR, strict=True)
protocol_lock = json.loads(
    (PROTOCOL_DIR / expcfg.PROTOCOL_OUTPUT_FILENAMES["lock"]).read_text(encoding="utf-8")
)
PROTOCOL_HASH = str(protocol_lock["lock_sha256"])

input_paths = pca_input_artifact_paths(PROJECT_ROOT, results_tag=RESULTS_TAG)
input_hashes = hash_pca_input_artifacts(PROJECT_ROOT, results_tag=RESULTS_TAG)
matrix_summary_df = pd.read_parquet(input_paths["matrix_summary"])
m_feasibility_df = pd.read_parquet(input_paths["m_feasibility"])
preprocessing_validation_df = pd.read_parquet(input_paths["preprocessing_validation"])
wavelength_config_df = pd.read_parquet(input_paths["wavelength_config"])
split_manifest_df = pd.read_parquet(input_paths["protocol_split_manifest"])

RESULTS_DIR = PROJECT_ROOT / "results" / f"{expcfg.PCA_RESULTS_DIR_PREFIX}_{RESULTS_TAG}"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATHS = {
    name: RESULTS_DIR / filename
    for name, filename in expcfg.PCA_OUTPUT_FILENAMES.items()
}
display(protocol_checks_df)
display(pd.DataFrame({"artifact": input_hashes.keys(), "sha256": input_hashes.values()}))


,check,passed,detail
0,all_frozen_artifacts_exist,True,missing=[]
1,configuration_sha256_matches_current_protocol,True,expected=d9a5771b0e6080b2b0f04da48f05540572694...
2,inference_plan_sha256_matches_current_protocol,True,expected=6f60368f60d7818e9555649769728ede68af1...
3,planned_contrasts_sha256_matches_current_protocol,True,expected=71d7fe422c54dbe6db8b3b640079541ea06e9...
4,checks_file_checksum_matches_lock,True,expected=2973b14a6b0732d2e656b2a8b107e020cc195...
5,inference_plan_file_checksum_matches_lock,True,expected=38757f08c4cf16549dc3ef17a869297f8af11...
6,manifest_file_checksum_matches_lock,True,expected=6f70f4ad603177f03e205ee786c008b32ffef...
7,planned_contrasts_file_checksum_matches_lock,True,expected=7a795bb2e75401ab033f33de3863bcd021352...
8,lock_checksum_is_valid,True,expected=40bc7403b904cc3936bc0f8f40f6cd8379f42...


,artifact,sha256
0,database_manifest,6db0157b3c5894c9ded5b0ca583dd202a0035b42a2c890...
1,protocol_split_manifest,8ece3008b943cb3cf12679ca06f8d64b04cf7daff996f9...
2,wavelength_config,7b652b14f1bff66e8ebaf75034ac585b23cd1d90ed2c15...
3,matrix_summary,ec091d20702adbf16ad5b909b161e6ec121af20dd1d220...
4,m_feasibility,408be1e526a588bbf75fd621bd7ffce7e396d1ca0b132e...
5,preprocessing_validation,4b4d86c9a97d04e87327edfbcebc97937fe418d2591462...


## 2. Axe spectral, univers de candidats et folds communs

In [3]:
object_db, image_db = load_nir_uco_h5(
    PROJECT_ROOT.joinpath(*expcfg.DATABASE_H5_RELATIVE_PATH),
    reconstruct_heavy_object_arrays=True,
)
if expcfg.USE_WAVELENGTH_WINDOW:
    object_db, image_db, wavelengths, _ = select_wavelength_range_from_database(
        object_db=object_db,
        image_db=image_db,
        min_nm=expcfg.WAVELENGTH_WINDOW_MIN_NM,
        max_nm=expcfg.WAVELENGTH_WINDOW_MAX_NM,
    )
else:
    wavelengths = np.asarray(next(iter(object_db.values()))["wavelengths"], dtype=float)

locked_axis_id = str(wavelength_config_df.iloc[0]["wavelength_axis_id"])
if not bool(wavelength_config_df.iloc[0]["locked"]):
    raise RuntimeError("Notebook 02 wavelength configuration is not locked.")
if wavelength_axis_id(wavelengths) != locked_axis_id:
    raise RuntimeError("Database wavelength axis differs from the notebook-02 lock.")

candidate_plan_df = build_pca_candidate_plan(
    matrix_summary_df,
    m_feasibility_df,
    preprocessing_validation_df,
    allowed_m=expcfg.PCA_BALANCED_M_VALUES,
    sg_window_length=expcfg.PCA_SG_WINDOW_LENGTH,
    matrix_methods=expcfg.PCA_MATRIX_METHODS,
    balanced_strategies=expcfg.PCA_BALANCED_STRATEGIES,
)
RUN_FINGERPRINT = build_pca_run_fingerprint(
    candidate_plan_df,
    protocol_hash=PROTOCOL_HASH,
    input_hashes=input_hashes,
)

calibration_object_ids = eligible_object_ids(split_manifest_df, "calibration")
object_db_calibration = subset_object_db_for_pca(
    object_db,
    sample_kind=expcfg.PCA_SAMPLE_KIND,
    reference_classes=expcfg.REFERENCE_CLASSES,
    allowed_batches=expcfg.PCA_CALIBRATION_BATCHES,
    forbidden_batches=expcfg.PCA_FORBIDDEN_BATCHES,
    allowed_object_ids=calibration_object_ids,
)
observed_batches = {int(obj["batch"]) for obj in object_db_calibration.values()}
if observed_batches != set(expcfg.PCA_CALIBRATION_BATCHES):
    raise RuntimeError(f"Unexpected PCA batches: {sorted(observed_batches)}")

calibration_manifest_df = split_manifest_df.loc[
    split_manifest_df["protocol_role"].eq("calibration")
    & split_manifest_df["qc_eligibility"].eq("accepted")
].copy()
common_folds_df, common_fold_diagnostics_df = build_grouped_folds(
    calibration_manifest_df,
    group_col=expcfg.PCA_STABILITY_GROUP_COL,
    label_col="label",
    batch_col="batch",
    n_splits=expcfg.PCA_STABILITY_N_SPLITS,
    random_state=expcfg.PCA_STABILITY_REFERENCE_SEED,
    require_complete_coverage=True,
)
display(candidate_plan_df.groupby(["matrix_variant"], as_index=False).agg(n_candidates=("candidate_id", "nunique")))
display(common_fold_diagnostics_df)


,matrix_variant,n_candidates
0,all_pixels,10
1,balanced_pixels_center_m10,10
2,balanced_pixels_center_m20,10
3,balanced_pixels_random_m10,10
4,balanced_pixels_random_m20,10
5,object_mean,19
6,object_median,19


,fold_id,n_validation_groups,n_training_groups,n_validation_objects,median_validation_object_size,validation_has_all_classes,validation_has_all_batches,training_has_all_classes,training_has_all_batches,no_shared_groups,coverage_complete
0,0,2,2,104,0.0,True,True,True,True,True,True
1,1,2,2,105,0.0,True,True,True,True,True,True


## 3. PCA et stabilité par candidat

In [4]:
diagnostic_rows = []
component_parts = []
candidate_results = {}
stability_registry = {}

for candidate in candidate_plan_df.to_dict("records"):
    diagnostic, result, components = fit_pca_candidate(
        object_db_calibration,
        candidate,
        n_components=expcfg.PCA_N_COMPONENTS,
        wavelengths=wavelengths,
        random_state=expcfg.PCA_STABILITY_REFERENCE_SEED,
        under_m_policy=expcfg.PCA_BALANCED_UNDER_M_POLICY,
        sg_window_length=expcfg.PCA_SG_WINDOW_LENGTH,
        sg_polyorder=expcfg.PCA_SG_POLYORDER,
        max_zero_variance_band_rate=expcfg.PREPROCESSING_MAX_ZERO_VARIANCE_BAND_RATE,
        zero_variance_epsilon=expcfg.PREPROCESSING_ZERO_VARIANCE_EPSILON,
    )
    diagnostic = dict(diagnostic)
    diagnostic["stability_valid"] = False
    if result is not None:
        candidate_results[candidate["candidate_id"]] = result
        component_parts.append(components)
        try:
            metric_stability, loading_stability = evaluate_pca_stability(
                object_db_calibration,
                candidate=candidate,
                fold_assignments=common_folds_df,
                group_col=expcfg.PCA_STABILITY_GROUP_COL,
                n_components=expcfg.PCA_STABILITY_N_COMPONENTS,
                seeds=expcfg.PCA_STABILITY_SEEDS,
                reference_seed=expcfg.PCA_STABILITY_REFERENCE_SEED,
                n_splits=expcfg.PCA_STABILITY_N_SPLITS,
                n_bootstrap=expcfg.PCA_STABILITY_N_BOOTSTRAP,
                bootstrap_group_col=expcfg.PCA_STABILITY_BOOTSTRAP_GROUP_COL,
                sg_window_length=expcfg.PCA_SG_WINDOW_LENGTH,
                sg_polyorder=expcfg.PCA_SG_POLYORDER,
                wavelengths=wavelengths,
                under_m_policy=expcfg.PCA_BALANCED_UNDER_M_POLICY,
            )
            diagnostic.update(summarize_pca_stability(metric_stability, loading_stability))
            stability_registry[candidate["candidate_id"]] = {
                "metrics": metric_stability,
                "loadings": loading_stability,
            }
        except Exception as error:
            diagnostic["stability_valid"] = False
            diagnostic["technical_error"] = (
                str(diagnostic.get("technical_error", ""))
                + f"; stability: {type(error).__name__}: {error}"
            ).strip("; ")
    else:
        diagnostic["stability_valid"] = False
    diagnostic_rows.append(dict(diagnostic))

pca_candidate_diagnostics_df = pd.DataFrame(diagnostic_rows)
technical_columns = [
    column for column in PCA_TECHNICAL_FLAG_COLUMNS
    if column in pca_candidate_diagnostics_df
]
technically_valid_mask = pca_candidate_diagnostics_df[technical_columns].fillna(False).all(axis=1)
review_candidates_df = pca_candidate_diagnostics_df.loc[technically_valid_mask].copy()
if review_candidates_df.empty:
    raise RuntimeError("No technically valid PCA candidate remains.")
review_candidate_ids = set(review_candidates_df["candidate_id"].astype(str))
review_plan_df = candidate_plan_df.loc[
    candidate_plan_df["candidate_id"].astype(str).isin(review_candidate_ids)
].copy()
display(pca_candidate_diagnostics_df.groupby(["matrix_family"], as_index=False).agg(n_candidates=("candidate_id", "size"), n_valid=("stability_valid", "sum")))


,matrix_family,n_candidates,n_valid
0,object_matrix,38,38
1,pixel_matrix,50,50


## 4. Dossier visuel exhaustif et revue humaine bloquante

Le PDF est généré avant la sélection, avec une page pour chaque candidat techniquement valide. Le tableau de revue est d’abord créé ou actualisé, puis la cellule suivante applique et valide les décisions documentées. La validation bloquante intervient seulement après cette saisie.


In [5]:
page_by_candidate = build_pca_visual_review_pdf(
    candidate_results,
    review_plan_df,
    OUTPUT_PATHS["visual_review"],
    wavelengths=wavelengths,
)
review_pdf_sha256 = sha256_file(OUTPUT_PATHS["visual_review"])
existing_review_df = (
    pd.read_parquet(OUTPUT_PATHS["artifact_review"])
    if OUTPUT_PATHS["artifact_review"].exists()
    else None
)
pca_artifact_review_df = build_pca_artifact_review_table(
    review_plan_df,
    run_fingerprint=RUN_FINGERPRINT,
    review_pdf_path=str(OUTPUT_PATHS["visual_review"].resolve()),
    review_pdf_sha256=review_pdf_sha256,
    page_by_candidate=page_by_candidate,
    existing_review=existing_review_df,
).loc[:, list(expcfg.PCA_ARTIFACT_REVIEW_COLUMNS)]
save_parquet(pca_artifact_review_df, OUTPUT_PATHS["artifact_review"])

display(pca_artifact_review_df["review_decision"].value_counts(dropna=False))


review_decision
    88
Name: count, dtype: int64

### Révision documentée du PDF courant

La cellule suivante rattache les décisions humaines au `run_fingerprint` du run
courant. Le seul verrou recopié après lecture humaine est
`REVIEWED_PDF_SHA256` : il doit être l'empreinte affichée pour le PDF relu.

Un changement du verrou de protocole n'impose donc pas une seconde lecture si
le PDF régénéré est strictement identique octet par octet. En revanche, toute
modification du PDF, de l'ordre des pages ou de l'univers des candidats bloque
la cellule et impose une nouvelle revue. Les candidats non cités dans les
groupes documentés sont explicitement classés `accept`.


In [6]:
REVIEWED_PDF_SHA256 = (
    "1e2a7950925019166c28813d380f40e72f3ebb1d732ef70877303eee2eaa942b"
)
reviewer_name = "AG"
review_date = "2026-08-03"

reject_msc = {
    "pca_65f0e6f494c80aad4c08",
    "pca_f6a95faaea69f4f457f6",
    "pca_97493d855c914065a5f5",
    "pca_73cc58020fe14131663e",
}
reject_snv = {
    "pca_c3caddf5814a1b306cfd",
    "pca_fa500179ea69f9dfb9f5",
    "pca_498e9cbf785d9e4ac5eb",
    "pca_0767aab422d9136719ff",
}
reject_snv_smooth = {
    "pca_2c58d46cd53cd67d1b9d",
    "pca_714f9cfb5b58de7aec46",
    "pca_3a4ebd49afddad8d4f6c",
    "pca_c1a231e41846ec757e98",
}
reject_vector_norm = {
    "pca_b417cc6a7ddfaecf04e0",
    "pca_3024b532232f7f520bb1",
    "pca_7190f7409e74c87d0548",
    "pca_deebeb8ac2d33fd6320d",
}
warning_snv_derivatives = {
    "pca_7ce95b6e6b369462420f",
    "pca_8213ce8cf9cbea6098a9",
    "pca_652ec16d5513824cb6a0",
    "pca_0f4bd9bde95557ac368a",
    "pca_0c0fd432430d9b4fc8f3",
    "pca_b925a981e85f6cfaef20",
    "pca_57908f1b2509320d4727",
    "pca_4a5c7fbabe45fd9cec0e",
    "pca_324794a1759b5bb131fa",
    "pca_6d002727bcfe3b39c6a7",
}
warning_msc_center_m10 = {"pca_cf74951687550ba1af86"}
warning_snv_center_m10 = {"pca_ad89d611764529a55b6e"}
warning_snv_smooth_center_m10 = {"pca_24fe7c8e1e38d5054486"}

decision_groups = (
    {
        "candidate_ids": reject_msc,
        "review_decision": "reject",
        "artifact_codes": (
            "msc_pixel_instability;extreme_score_outlier;extreme_t2_leverage"
        ),
        "critical_artifact": True,
        "review_comment": (
            "Quelques observations extrêmes dominent les scores PCA et T² ; "
            "le nuage principal est comprimé. MSC est visuellement instable "
            "pour cette matrice pixel."
        ),
    },
    {
        "candidate_ids": reject_snv,
        "review_decision": "reject",
        "artifact_codes": (
            "snv_outlier_amplification;score_cloud_compression;qt2_outliers"
        ),
        "critical_artifact": True,
        "review_comment": (
            "SNV amplifie des observations pixel atypiques ; de longues queues "
            "apparaissent dans les scores et les distances Q–T², au détriment "
            "de la représentation du nuage principal."
        ),
    },
    {
        "candidate_ids": reject_snv_smooth,
        "review_decision": "reject",
        "artifact_codes": (
            "snv_outlier_amplification;smoothed_outlier_persistence;qt2_outliers"
        ),
        "critical_artifact": True,
        "review_comment": (
            "Le lissage ne corrige pas l'instabilité introduite par SNV ; les "
            "observations extrêmes persistent et influencent fortement les axes PCA."
        ),
    },
    {
        "candidate_ids": reject_vector_norm,
        "review_decision": "reject",
        "artifact_codes": (
            "vector_norm_leverage_outlier;score_cloud_compression;"
            "extreme_t2_leverage"
        ),
        "critical_artifact": True,
        "review_comment": (
            "La normalisation vectorielle produit une observation de leverage très "
            "isolée qui impose l'échelle des scores et de T² ; la PCA n'est plus "
            "représentative du nuage principal."
        ),
    },
    {
        "candidate_ids": warning_snv_derivatives,
        "review_decision": "warning",
        "artifact_codes": (
            "snv_derivative_residual_outliers;moderate_qt2_outliers"
        ),
        "critical_artifact": False,
        "review_comment": (
            "Quelques observations atypiques et résidus Q–T² persistent après "
            "SNV et dérivation, mais ils ne déterminent pas entièrement les axes "
            "et le nuage principal reste interprétable."
        ),
    },
    {
        "candidate_ids": warning_msc_center_m10,
        "review_decision": "warning",
        "artifact_codes": (
            "msc_pixel_instability;isolated_score_outlier;moderate_qt2_outliers"
        ),
        "critical_artifact": False,
        "review_comment": (
            "Une observation isolée est visible après MSC, mais le nuage principal "
            "reste lisible et la représentation PCA n'est pas entièrement dominée."
        ),
    },
    {
        "candidate_ids": warning_snv_center_m10,
        "review_decision": "warning",
        "artifact_codes": "snv_outlier_amplification;moderate_qt2_outliers",
        "critical_artifact": False,
        "review_comment": (
            "Une amplification modérée d'observations atypiques est visible après "
            "SNV, sans écrasement complet du nuage principal."
        ),
    },
    {
        "candidate_ids": warning_snv_smooth_center_m10,
        "review_decision": "warning",
        "artifact_codes": (
            "snv_outlier_amplification;smoothed_outlier_persistence;"
            "moderate_qt2_outliers"
        ),
        "critical_artifact": False,
        "review_comment": (
            "Des observations atypiques persistent après SNV et lissage, mais la "
            "structure centrale des scores demeure interprétable."
        ),
    },
)

review_path = OUTPUT_PATHS["artifact_review"]
review = pd.read_parquet(review_path).copy()
expected_candidate_ids = set(review_plan_df["candidate_id"].astype(str))
observed_candidate_ids = set(review["candidate_id"].astype(str))
if observed_candidate_ids != expected_candidate_ids:
    raise RuntimeError(
        "Le fichier de revue ne correspond pas au run courant : "
        f"missing={sorted(expected_candidate_ids - observed_candidate_ids)}, "
        f"extra={sorted(observed_candidate_ids - expected_candidate_ids)}"
    )
if not review["run_fingerprint"].astype(str).eq(RUN_FINGERPRINT).all():
    raise RuntimeError(
        "Le tableau de revue n'est pas rattaché au run_fingerprint courant."
    )

pca_artifact_review_df = apply_pca_artifact_review_decisions(
    review,
    decision_groups=decision_groups,
    reviewed_pdf_sha256=REVIEWED_PDF_SHA256,
    reviewer=reviewer_name,
    review_date=review_date,
    default_review_comment=(
        "Spectres et loadings cohérents ; nuage des scores interprétable et non "
        "dominé par des observations isolées. Aucun artefact visuel critique."
    ),
).loc[:, list(expcfg.PCA_ARTIFACT_REVIEW_COLUMNS)]

decision_counts = pca_artifact_review_df["review_decision"].value_counts().to_dict()
n_reject = sum(
    len(group["candidate_ids"])
    for group in decision_groups
    if group["review_decision"] == "reject"
)
n_warning = sum(
    len(group["candidate_ids"])
    for group in decision_groups
    if group["review_decision"] == "warning"
)
expected_counts = {
    "accept": len(review) - n_reject - n_warning,
    "warning": n_warning,
    "reject": n_reject,
}
if decision_counts != expected_counts:
    raise RuntimeError(f"Comptage inattendu : {decision_counts} != {expected_counts}")

validate_pca_artifact_review(
    pca_artifact_review_df,
    expected_candidate_ids=review_plan_df["candidate_id"],
    expected_run_fingerprint=RUN_FINGERPRINT,
)
save_parquet(pca_artifact_review_df, review_path)

display(
    pca_artifact_review_df[
        ["review_decision", "critical_artifact"]
    ].value_counts(dropna=False)
)
print("run_fingerprint courant :", RUN_FINGERPRINT)
print("SHA-256 du PDF relu :", REVIEWED_PDF_SHA256)
print(f"Revue enregistrée dans : {review_path.resolve()}")


review_decision  critical_artifact
accept           False                59
reject           True                 16
warning          False                13
Name: count, dtype: int64

run_fingerprint courant : 77fd5b7edbb7f5e8a91c6daca242184945f57710d0c8183472d7842663e23231
SHA-256 du PDF relu : 1e2a7950925019166c28813d380f40e72f3ebb1d732ef70877303eee2eaa942b
Revue enregistrée dans : C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\03_pca_non_noisy_all\pca_artifact_review.parquet


## 5. Couverture stricte, agrégation robuste et Pareto par famille

In [7]:
selection_config = expcfg.make_pca_selection_config()
pca_candidate_selection_diagnostics_df = build_pca_selection_diagnostics(
    pca_candidate_diagnostics_df,
    artifact_review_df=pca_artifact_review_df,
    config=selection_config,
)
(
    pca_shortlist_full_df,
    pca_preprocessing_summary_df,
    pca_selection_stage_summary_df,
) = select_pca_preprocessing_shortlist(
    pca_candidate_selection_diagnostics_df,
    config=selection_config,
)

review_hash = hash_pca_review_table(pca_artifact_review_df)
pca_shortlist_frozen_df = freeze_pca_shortlist(
    pca_shortlist_full_df,
    protocol_hash=PROTOCOL_HASH,
    review_hash=review_hash,
    input_hashes=input_hashes,
)
pca_selected_preprocessings_df = pca_shortlist_frozen_df.loc[
    :, list(expcfg.PCA_SELECTED_PREPROCESSING_COLUMNS)
].copy()
validate_pca_preprocessing_shortlist(
    pca_selected_preprocessings_df,
    max_per_family=expcfg.MAX_PCA_PREPROCESSINGS_PER_FAMILY,
    expected_families=expcfg.PCA_SELECTION_EXPECTED_FAMILIES,
    expected_protocol_hash=PROTOCOL_HASH,
)

pca_scoring_diagnostics_df = build_pca_scoring_diagnostics(
    pca_candidate_selection_diagnostics_df,
    config=selection_config,
    preprocessing_summary_df=pca_preprocessing_summary_df,
).loc[:, list(expcfg.PCA_SCORING_DIAGNOSTIC_COLUMNS)]
pca_components_df = pd.concat(component_parts, ignore_index=True)
pca_summary_df = pca_components_df.loc[
    pca_components_df["candidate_id"].astype(str).isin(review_candidate_ids),
    list(expcfg.PCA_SUMMARY_COLUMNS),
].copy()
display(pca_selection_stage_summary_df)
display(pca_preprocessing_summary_df)
display(pca_selected_preprocessings_df)


,matrix_family,stage,n_entering,n_retained,n_eliminated,retention_rate
0,object_matrix,input,19,19,0,1.000000
1,object_matrix,strict_family_coverage,19,19,0,1.000000
2,object_matrix,complete_pareto_metrics,19,19,0,1.000000
3,object_matrix,pareto_front,19,13,6,0.684211
4,pixel_matrix,input,10,10,0,1.000000
5,pixel_matrix,strict_family_coverage,10,6,4,0.600000
6,pixel_matrix,complete_pareto_metrics,6,6,0,1.000000
7,pixel_matrix,pareto_front,6,4,2,0.666667


,matrix_family,preprocessing,preprocessing_steps,n_candidates,n_expected_variants,n_observed_variants,expected_variants_json,observed_variants_json,missing_variants_json,extra_variants_json,candidate_ids_json,n_accept,n_warning,n_reject,n_blocked_candidates,coverage_complete,all_candidates_admissible,strict_coverage_pass,objective_metrics_complete,preprocessing_eligible,...,instability_metric_median,instability_metric_max,instability_metric_iqr,instability_metric_worst,ncomp_95_min,ncomp_95_median,ncomp_95_max,ncomp_95_iqr,ncomp_95_worst,object_class_trace_ratio_min,object_class_trace_ratio_median,object_class_trace_ratio_max,object_class_trace_ratio_iqr,object_class_trace_ratio_worst,object_batch_trace_ratio_min,object_batch_trace_ratio_median,object_batch_trace_ratio_max,object_batch_trace_ratio_iqr,object_batch_trace_ratio_worst,selection_reason
0,object_matrix,absorbance,absorbance,2,2,2,"[""object_mean"", ""object_median""]","[""object_mean"", ""object_median""]",[],[],"[""pca_876b94b7d554d6dffbbc"", ""pca_a570c44fbca3...",2,0,0,0,True,True,True,True,True,...,0.029719,0.048825,0.019106,0.048825,1.0,1.5,2.0,0.5,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,strict_family_coverage;preprocessing_level_par...
1,object_matrix,absorbance_msc,absorbance+msc,2,2,2,"[""object_mean"", ""object_median""]","[""object_mean"", ""object_median""]",[],[],"[""pca_3e9f18cf1452652c43ed"", ""pca_6b1776dbad81...",2,0,0,0,True,True,True,True,True,...,0.051356,0.068825,0.017470,0.068825,5.0,10.0,15.0,5.0,15.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,strict_family_coverage;preprocessing_level_par...
2,object_matrix,absorbance_sg_d1,absorbance+sg_d1,2,2,2,"[""object_mean"", ""object_median""]","[""object_mean"", ""object_median""]",[],[],"[""pca_760f4229253f9720e3ca"", ""pca_b956e355d9b5...",2,0,0,0,True,True,True,True,True,...,0.012824,0.021594,0.008770,0.021594,3.0,3.0,3.0,0.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,strict_family_coverage;preprocessing_level_par...
3,object_matrix,absorbance_sg_d2,absorbance+sg_d2,2,2,2,"[""object_mean"", ""object_median""]","[""object_mean"", ""object_median""]",[],[],"[""pca_d11dd6677accf3e305f1"", ""pca_ee8bb3693271...",2,0,0,0,True,True,True,True,True,...,0.061298,0.085085,0.023787,0.085085,4.0,4.5,5.0,0.5,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,dominated_on_preprocessing_pareto_by:snv_sg_d1...
4,object_matrix,absorbance_sg_smooth,absorbance+sg_smooth,2,2,2,"[""object_mean"", ""object_median""]","[""object_mean"", ""object_median""]",[],[],"[""pca_28d83fb64174c1700a31"", ""pca_3cecbe8a289e...",2,0,0,0,True,True,True,True,True,...,0.009577,0.012833,0.003257,0.012833,1.0,1.5,2.0,0.5,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,strict_family_coverage;preprocessing_level_par...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24,pixel_matrix,snv,snv,5,5,5,"[""all_pixels"", ""balanced_pixels_center_m10"", ""...","[""all_pixels"", ""balanced_pixels_center_m10"", ""...",[],[],"[""pca_0767aab422d9136719ff"", ""pca_498e9cbf785d...",0,1,4,4,True,False,False,True,False,...,0.034355,0.101394,0.024464,0.101394,10.0,12.0,26.0,12.0,26.0,0.162628,0.244937,0.290103,0.084877,0.162628,0.000316,0.002331,0.007432,0.005332,0.007432,ineligible:missing_variant_or_blocked_candidate
25,pixel_matrix,snv_sg_d1,snv+sg_d1,5,5,5,"[""all_pixels"", ""balanced_pixels_center_m10"", ""...","[""all_pixels"", ""balanced_pixels_center_m10"", ""...",[],[],"[""pca_0c0fd432430d9b4fc8f3"", ""pca_324794a1759b...",0,5,0,0,True,True,True,True,True,...,0.005129,0.041086,0.007709,0.041086,4.0,4.0,4.0,0.0,4.0,0.078926,0.113117,0.142778,0.031470,0.078926,0.000226,0.002958,0.010279,0.005455,0.010279,dominated_on_preprocessing_pareto_by:sg_smooth
26,pixel_matrix,snv_sg_d2,snv+sg_d2,5,5,5,"[""all_pixels"", ""balanced_pixels_center_m10"", ""...","[""all_pixels"", ""balanced_pixels_center_m10"", ""...",[],[],"[""pca_0f4bd9bde95557ac368a"", "

,shortlist_id,protocol_hash,input_fingerprint,review_hash,matrix_family,preprocessing,preprocessing_steps,selection_status,selection_reason
0,pca_shortlist_9aa62f470cb77365d530,40bc7403b904cc3936bc0f8f40f6cd8379f42e7e170253...,a4eaf3a708cc040fe6d35d4f581f323a4079b54c09404b...,ede6aabae010e5eb592139fc720b1b530d4521d292a7b7...,object_matrix,absorbance,absorbance,selected,strict_family_coverage;preprocessing_level_par...
1,pca_shortlist_9aa62f470cb77365d530,40bc7403b904cc3936bc0f8f40f6cd8379f42e7e170253...,a4eaf3a708cc040fe6d35d4f581f323a4079b54c09404b...,ede6aabae010e5eb592139fc720b1b530d4521d292a7b7...,object_matrix,absorbance_msc,absorbance+msc,selected,strict_family_coverage;preprocessing_level_par...
2,pca_shortlist_9aa62f470cb77365d530,40bc7403b904cc3936bc0f8f40f6cd8379f42e7e170253...,a4eaf3a708cc040fe6d35d4f581f323a4079b54c09404b...,ede6aabae010e5eb592139fc720b1b530d4521d292a7b7...,object_matrix,absorbance_sg_d1,absorbance+sg_d1,selected,strict_family_coverage;preprocessing_level_par...
3,pca_shortlist_9aa62f470cb77365d530,40bc7403b904cc3936bc0f8f40f6cd8379f42e7e170253...,a4eaf3a708cc040fe6d35d4f581f323a4079b54c09404b...,ede6aabae010e5eb592139fc720b1b530d4521d292a7b7...,object_matrix,absorbance_sg_smooth,absorbance+sg_smooth,selected,strict_family_coverage;preprocessing_level_par...
4,pca_shortlist_9aa62f470cb77365d530,40bc7403b904cc3936bc0f8f40f6cd8379f42e7e170253...,a4eaf3a708cc040fe6d35d4f581f323a4079b54c09404b...,ede6aabae010e5eb592139fc720b1b530d4521d292a7b7...,object_matrix,absorbance_snv,absorbance+snv,selected,strict_family_coverage;preprocessing_level_par...
5,pca_shortlist_9aa62f470cb77365d530,40bc7403b904cc3936bc0f8f40f6cd8379f42e7e170253...,a4eaf3a708cc040fe6d35d4f581f323a4079b54c09404b...,ede6aabae010e5eb592139fc720b1b530d4521d292a7b7...,object_matrix,absorbance_snv_sg_d1,absorbance+snv+sg_d1,selected,strict_family_coverage;preprocessing_level_par...
6,pca_shortlist_9aa62f470cb77365d530,40bc7403b904cc3936bc0f8f40f6cd8379f42e7e170253...,a4eaf3a708cc040fe6d35d4f581f323a4079b54c09404b...,ede6aabae010e5eb592139fc720b1b530d4521d292a7b7...,object_matrix,absorbance_snv_sg_smooth,absorbance+snv+sg_smooth,selected,strict_family_coverage;preprocessing_level_par...
7,pca_shortlist_9aa62f470cb77365d530,40bc7403b904cc3936bc0f8f40f6cd8379f42e7e170253...,a4eaf3a708cc040fe6d35d4f581f323a4079b54c09404b...,ede6aabae010e5eb592139fc720b1b530d4521d292a7b7...,object_matrix,msc,msc,selected,strict_family_coverage;preprocessing_level_par...
8,pca_shortlist_9aa62f470cb77365d530,40bc7403b904cc3936bc0f8f40f6cd8379f42e7e170253...,a4eaf3a708cc040fe6d35d4f581f323a4079b54c09404b...,ede6aabae010e5eb592139fc720b1b530d4521d292a7b7...,object_matrix,sg_smooth,sg_smooth,selected,strict_family_coverage;preprocessing_level_par...
9,pca_shortlist_9aa62f470cb77365d530,40bc7403b904cc3936bc0f8f40f6cd8379f42e7e170253...,a4eaf3a708cc040fe6d35d4f581f323a4079b54c09404b...,ede6aabae010e5eb592139fc720b1b530d4521d292a7b7...,object_matrix,snv,snv,selected,strict_family_coverage;preprocessing_level_par...


## 6. Gel des cinq tables et audit final

In [8]:
score_column_audit_df = assert_no_forbidden_score_columns(
    {
        "pca_summary": pca_summary_df,
        "pca_scoring_diagnostics": pca_scoring_diagnostics_df,
        "pca_preprocessing_summary": pca_preprocessing_summary_df,
        "pca_selected_preprocessings": pca_selected_preprocessings_df,
        "pca_artifact_review": pca_artifact_review_df,
    }
)
if any("confirmation" in column.lower() for column in pca_scoring_diagnostics_df.columns):
    raise RuntimeError("External-batch diagnostics are forbidden in notebook 03.")

save_parquet(pca_summary_df, OUTPUT_PATHS["summary"])
save_parquet(pca_scoring_diagnostics_df, OUTPUT_PATHS["diagnostics"])
save_parquet(pca_preprocessing_summary_df, OUTPUT_PATHS["preprocessing_summary"])
save_parquet(pca_selected_preprocessings_df, OUTPUT_PATHS["selected"])
save_parquet(pca_artifact_review_df, OUTPUT_PATHS["artifact_review"])

saved_shortlist_df = pd.read_parquet(OUTPUT_PATHS["selected"])
validate_pca_preprocessing_shortlist(
    saved_shortlist_df,
    max_per_family=expcfg.MAX_PCA_PREPROCESSINGS_PER_FAMILY,
    expected_families=expcfg.PCA_SELECTION_EXPECTED_FAMILIES,
    expected_protocol_hash=PROTOCOL_HASH,
)
output_hashes_df = pd.DataFrame(
    [
        {"artifact": name, "path": str(path), "sha256": sha256_file(path)}
        for name, path in OUTPUT_PATHS.items()
    ]
)
display(score_column_audit_df)
display(output_hashes_df)
print("shortlist_id:", saved_shortlist_df["shortlist_id"].iloc[0])
print("protocol_hash:", PROTOCOL_HASH)
print("run_fingerprint:", RUN_FINGERPRINT)


,table,n_columns,forbidden_score_columns,score_free
0,pca_summary,11,,True
1,pca_scoring_diagnostics,19,,True
2,pca_preprocessing_summary,54,,True
3,pca_selected_preprocessings,9,,True
4,pca_artifact_review,16,,True


,artifact,path,sha256
0,summary,C:\Users\alixg\OneDrive - Université Paris-Dau...,2780b0294f59e01f01b6e2da81f3a86ec0a12fba93afa9...
1,diagnostics,C:\Users\alixg\OneDrive - Université Paris-Dau...,34cfba7b0cdd38aa32bb50777451984f87894a8ee4b28c...
2,preprocessing_summary,C:\Users\alixg\OneDrive - Université Paris-Dau...,3cefc9a0b0a64586828e909e0d9cbe259d329acbe80cde...
3,selected,C:\Users\alixg\OneDrive - Université Paris-Dau...,a3bf7636cbe1e0a3d03fa8a8058a7207a325d0da14adae...
4,artifact_review,C:\Users\alixg\OneDrive - Université Paris-Dau...,5cc33b8295a6c49504d34619a4878f25a2f574c5b2c56f...
5,visual_review,C:\Users\alixg\OneDrive - Université Paris-Dau...,1e2a7950925019166c28813d380f40e72f3ebb1d732ef7...


shortlist_id: pca_shortlist_9aa62f470cb77365d530
protocol_hash: 40bc7403b904cc3936bc0f8f40f6cd8379f42e7e170253e1f09eb0c673f66601
run_fingerprint: 77fd5b7edbb7f5e8a91c6daca242184945f57710d0c8183472d7842663e23231
